# einops-repeat — worked example 1: Broadcast a per-feature bias vector across batch and time dimensions

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-repeat`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import torch.nn.functional as F
import einops
from einops import repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`einops.repeat` introduces new axes or replicates existing ones without copying the underlying data (it creates a stride-0 view). The pattern `'d -> b t d'` with `b=B, t=T` broadcasts a 1-D feature bias `(D,)` into a `(B, T, D)` tensor, producing the same bias values at every batch position and timestep.

## Worked solution

Input: `bias` of shape `(D=16,)` — one scalar bias per feature.

**Pattern:** `'d -> b t d'` with `b=4, t=8`.

Einops introduces new axes `b` and `t` ahead of the existing `d` axis. The resulting tensor has shape `(4, 8, 16)`. Because it is a stride-0 view, `bias[f]` is accessible at every `bias_b[i, j, f]` without copying.

**Use case:** adding a position-independent bias to every token embedding in a Transformer — `logits + bias_b` broadcasts the bias across all batch positions and sequence positions.

In [ ]:
import torch as t
from einops import repeat

t.manual_seed(10)
D, B, T = 16, 4, 8
bias = t.randn(D)

def broadcast_feature_bias(bias, B, T):
    return repeat(bias, 'd -> b t d', b=B, t=T)

bias_b = broadcast_feature_bias(bias, B, T)
print('Bias shape:', bias.shape)    # (16,)
print('Broadcast shape:', bias_b.shape)  # (4, 8, 16)
assert bias_b.shape == (B, T, D)

# Verify values: every (b, t, :) slice equals the original bias
assert t.allclose(bias_b[0, 0], bias)
assert t.allclose(bias_b[3, 7], bias)

# Stride-0 on introduced axes (no data copy)
print('Strides:', bias_b.stride())  # (0, 0, 1) for b, t axes
assert bias_b.data_ptr() == bias.data_ptr()
print('No copy — shares storage with original bias:', True)